# 01 Explore a trajectory dataset

Generates a small dataset if none is present, then looks at the schema, outcomes, simulator phase labels and force traces. Everything shown is computed from the generated data; nothing is precomputed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from contact_state_reuse.analysis.plotting import plot_episode_overview
from contact_state_reuse.data import DatasetConfig, generate_dataset, load_dataset
from contact_state_reuse.envs import MODE_NAMES

path = Path("../data/notebook/explore_source.npz")
if not path.exists():
    generate_dataset(DatasetConfig(name="explore_source", domain="source", n_episodes=60, seed=0, out_dir=str(path.parent)))
data = load_dataset(path)
a = data.arrays
print(data.n_episodes, "episodes,", len(a["t"]), "rows; layout:", data.meta["obs_layout"])

## Outcomes and expert variants

In [ ]:
eps = pd.DataFrame({k: a[k] for k in ("ep_domain", "ep_variant", "ep_success", "ep_jam", "ep_length")})
eps.groupby("ep_variant")[["ep_success", "ep_jam", "ep_length"]].mean().round(2)

## Simulator phase labels

These labels are computed from privileged simulator state for analysis. They are not policy inputs.

In [ ]:
counts = pd.Series(a["contact_mode"]).value_counts().sort_index()
counts.index = [MODE_NAMES[i] for i in counts.index]
counts

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
modes = a["contact_mode"]
f = a["contact_force_norm"]
ax.boxplot([f[modes == k] for k in range(len(MODE_NAMES))], tick_labels=MODE_NAMES, showfliers=False)
ax.set_ylabel("true contact force [N]")
ax.set_title("contact force by simulator phase label")
plt.show()

## Force and depth traces

In [ ]:
out = plot_episode_overview(data, "../outputs/notebook/dataset_episodes.png", n_episodes=3)
from IPython.display import Image
Image(str(out))

## Observed vs true force

The policy sees the force channel after the sensor model (noise, bias, delay, dropout).

In [ ]:
obs_f = a["obs"][:, data.layout["force"]]
true_f = a["wrench_true"]
err = obs_f[:, :2] - true_f[:, :2]
print("observed minus true force: mean", err.mean(0).round(3), "std", err.std(0).round(3))